In [ ]:
pip install pandas openpyxl tqdm


[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip



  Using cached openpyxl-3.1.5-py2.py3-none-any.whl.metadata (2.5 kB)
  Using cached tqdm-4.67.3-py3-none-any.whl.metadata (57 kB)
  Using cached et_xmlfile-2.0.0-py3-none-any.whl.metadata (2.7 kB)
Using cached openpyxl-3.1.5-py2.py3-none-any.whl (250 kB)
Using cached tqdm-4.67.3-py3-none-any.whl (78 kB)
Using cached et_xmlfile-2.0.0-py3-none-any.whl (18 kB)


In [5]:
import os
import pickle
import pandas as pd
from pathlib import Path
from tqdm import tqdm # 换成最普通的 tqdm

# ================= 配置区 =================
# 指向你的 result 根目录，例如 './result/archive-27'
ROOT_DIR = "../result/archive-98" 
OUTPUT_FILE_LOC = "./result_98_2/experiment_results_PowerBI.csv"
# ==========================================

def process_experiment_data(root_path):
    root_path = Path(root_path)
    data_records = []
    
    # 递归查找所有的 .pckl 文件
    # 使用 rglob 自动穿透 task/num/seed 这几层目录
    pckl_files = list(root_path.rglob("*.pckl"))
    
    print(f"🔍 寻找到 {len(pckl_files)} 个 pickle 文件，开始解析...")
    
    for pckl_file in tqdm(pckl_files, desc="Parsing Files"):
        try:
            # 1. 从目录结构解析环境参数 (root/task/num/seed/file.pckl)
            # 通过 parent 逐级向上获取文件夹名称
            seed_str = pckl_file.parent.name
            num_str = pckl_file.parent.parent.name
            task_name = pckl_file.parent.parent.parent.name
            
            # 2. 从文件名解析策略参数
            filename = pckl_file.name.replace('.pckl', '')
            parts = filename.split('-')
            
            # 确保文件名是我们预期的格式 (防崩溃)
            if len(parts) >= 7 and parts[0] == 'EfficientBFS':
                alg = parts[0]
                strategy_raw = parts[1]
                heuristic = parts[2]
                sorting = parts[3]
                budget = float(parts[4])
                alpha = float(parts[5])
                model_name = parts[6]
                
                # 3. 剥离并识别 Local Search 标记
                is_ls = strategy_raw.endswith('_LS')
                strategy_clean = strategy_raw.replace('_LS', '') if is_ls else strategy_raw
                
                # 4. 读取 Pickle 内容
                with open(pckl_file, 'rb') as f:
                    res = pickle.load(f)
                    
                # 5. 组装 PowerBI 友好的扁平字典
                record = {
                    'Task': task_name,
                    'Ground_Size': int(num_str),
                    'Seed': int(seed_str),
                    'Algorithm': alg,
                    'Strategy': strategy_clean,
                    'Local_Search': is_ls,      # True/False，在 PowerBI 里可以直接做切片器
                    'Heuristic': heuristic,
                    'Sorting': sorting,
                    'Budget': budget,
                    'Alpha': alpha,
                    'Model': model_name,
                    
                    # 提取核心指标 (使用 .get 防御性编程，防止某个文件不完整)
                    'Objective_f(S)': res.get('f(S)', None),
                    'Cost_c(S)': res.get('c(S)', None),
                    'Time_s': res.get('time', None),
                    'Node_Count': res.get('node_count', None),
                    'Open_List_Count': res.get('open_list_count', None),
                    'TLE': res.get('TLE', False),
                
                    # 不要把具体的集合 S 导进去，Excel 存大列表很灾难，存一下长度作为特征即可
                    'Solution_Set_Size': len(res.get('S', [])) if 'S' in res else 0 
                }
                
                data_records.append(record)
                
        except Exception as e:
            # 记录解析失败的文件，但不中断整个循环
            print(f"❌ 解析出错 {pckl_file.name}: {e}")

    # 6. 转换为 DataFrame 并导出
    df = pd.DataFrame(data_records)
    
    # 按照 Budget, Strategy, Local_Search 排序，让表格看起来更整洁
    if not df.empty:
        df.sort_values(by=['Task', 'Budget', 'Strategy', 'Local_Search', 'Seed'], inplace=True)
        # df.to_excel(OUTPUT_EXCEL, index=False)
        df.to_csv(OUTPUT_FILE_LOC, index=False, encoding='utf-8-sig')
        print(f"\n✅ 数据处理完毕！共提取 {len(df)} 条有效记录。")
        print(f"💾 已保存至: {OUTPUT_EXCEL}")
    else:
        print("\n⚠️ 未提取到任何有效数据，请检查 ROOT_DIR 路径是否正确。")
        
    return df

# 执行提取
df_results = process_experiment_data(ROOT_DIR)

# 预览前 5 行数据
display(df_results.head())

🔍 寻找到 402 个 pickle 文件，开始解析...


Parsing Files: 100%|██████████| 402/402 [00:01<00:00, 232.99it/s]


✅ 数据处理完毕！共提取 401 条有效记录。
💾 已保存至: ./result_98/experiment_results_PowerBI.xlsx


,Task,Ground_Size,Seed,Algorithm,Strategy,Local_Search,Heuristic,Sorting,Budget,Alpha,Model,Objective_f(S),Cost_c(S),Time_s,Node_Count,Open_List_Count,TLE,Solution_Set_Size
21,adult,111,0,EfficientBFS,density_gap,False,ub2,d,6.0,0.95,AdultIncomeFeatureSelection,7.206555,5.836918,29.113479,5,1,False,5
46,adult,111,0,EfficientBFS,fullbab,False,ub2,d,6.0,0.95,AdultIncomeFeatureSelection,7.206555,5.836918,37.456656,12,1,False,5
71,adult,111,0,EfficientBFS,look_ahead,False,ub2,d,6.0,0.95,AdultIncomeFeatureSelection,7.206555,5.836918,30.124027,4,1,False,5
96,adult,111,0,EfficientBFS,traditional,False,ub2,d,6.0,0.95,AdultIncomeFeatureSelection,7.206555,5.836918,32.139664,5,1,False,5
22,adult,111,0,EfficientBFS,density_gap,False,ub2,d,7.0,0.95,AdultIncomeFeatureSelection,7.206555,6.999571,59.563825,22,1,False,6
